In [1]:
import numpy as np
import pandas as pd
import glob, os, subprocess, vcf, pysam, shutil, sparse, yaml, sys, pickle, itertools
from Bio import Entrez, Seq, SeqIO, SeqUtils
import scipy.stats as st
import evcouplings

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
model_loci = pd.read_csv("../data_utils/drug_loci.csv")

BASE_TO_COLUMN = {'A': 0, 'C': 1, 'T': 2, 'G': 3, '-': 4}
data_dir = "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs"
output_dir = "/n/data1/hms/dbmi/farhat/Sanjana/CNN_results"

h37Rv = SeqIO.read("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/GCF_000195955.2_ASM19595v2_genomic.gbff", "genbank")
h37Rv_full = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/mycobrowser_h37rv_v4.csv")
h37Rv_genes = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/mycobrowser_h37rv_genes_v4.csv")
h37Rv_coords_to_gene = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/h37Rv_coords_to_gene.csv")
h37Rv_coords_to_gene_dict = dict(zip(h37Rv_coords_to_gene['pos'], h37Rv_coords_to_gene['region']))

cc_df = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/criticalConcentrations_updated.csv")
# drug_gene_mapping = pd.read_csv("~/who-analysis/data/drug_gene_mapping.csv")

# CETR_isolate_metadata = pd.read_csv("../data_cleaning/CETR_isolate_details.csv")
# isolate_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/isolate_metadata.csv")
# print(set(h37Rv_genes.Symbol).symmetric_difference(h37Rv_full.query("Feature=='CDS'").Name))

results_dir = "/n/data1/hms/dbmi/farhat/Sanjana/CNN_results"

who_variants = pd.read_csv("../data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)
regression_who_results = pd.read_csv("/home/sak0914/who-analysis/results/Nov2024_Tier1.csv")
cc_df_who = pd.read_csv("/home/sak0914/who-analysis/data/drug_CC.csv")

coll_2014 = pd.read_csv("/home/sak0914/who-analysis/data/coll2014_SNP_scheme.tsv", sep="\t")
coll_2014["lineage"] = coll_2014["#lineage"].str.replace("lineage", "")
del coll_2014["#lineage"]

isolate_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/isolate_metadata.csv")

isolates_who_catalog_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/isolate_WHO_catalog_variants.csv")

lineages_matrix = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/lineage_matrix_Coll2014.csv", index_col=[0])

drug_abbr_dict = {"Delamanid": "DLM",
                  "Bedaquiline": "BDQ",
                  "Clofazimine": "CFZ",
                  "Ethionamide": "ETO",
                  "Linezolid": "LZD",
                  "Moxifloxacin": "MXF",
                  "Capreomycin": "CAP",
                  "Amikacin": "AMK",
                  "Pretomanid": "PMD",
                  "Pyrazinamide": "PZA",
                  "Kanamycin": "KAN",
                  "Levofloxacin": "LFX",
                  "Streptomycin": "STM",
                  "Ethambutol": "EMB",
                  "Isoniazid": "INH",
                  "Rifampicin": "RIF"
                 }

abbr_drug_dict = {value: key for key, value in drug_abbr_dict.items()}

df_samples_geno = pd.read_csv("../samples_pass_geno_QC.csv")
df_samples_geno['log_F2'] = np.log2(df_samples_geno['F2'])

genomic_data_dir = "/n/data1/hms/dbmi/farhat/rollingDB/genomic_data"

/tmp/ipykernel_2656/3954916128.py:32: DtypeWarning: Columns (36,37,99,100,102,103,106,108,112) have mixed types. Specify dtype option on import or set low_memory=False.
  who_variants = pd.read_csv("../data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)


In [5]:
isolate_metadata.query("OtherNames in ['C021', 'IDR1900043326', 'YA00085733', 'C153', 'IDR1700027275', 'YA00168159']")[['OtherNames', 'ROLLINGDB_ID', 'ISOLATION_COUNTRY', 'ISOLATION_REGION', 'ISOLATION_DATE']]

,OtherNames,ROLLINGDB_ID,ISOLATION_COUNTRY,ISOLATION_REGION,ISOLATION_DATE
725,C021,SAMN13813720,South Africa,South Africa,2019
827,C153,SAMN13813938,South Africa,South Africa,2019
72984,YA00085733,SAMN37124263,South Africa,Port Elizabeth,2017
73029,YA00168159,SAMN37124290,South Africa,Port Elizabeth,2019
73047,IDR1900043326,SAMN27762525,USA,New York,2019-12-23
73056,IDR1700027275,SAMN28552982,USA,Suffolk County,2017-07-05


In [7]:
df_BAM = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMEA2534701/bam/SAMEA2534701.depth.tsv.gz", sep='\t', compression='gzip', header=None)

In [15]:
len(df_BAM.loc[df_BAM[2] >= 10]) / len(df_BAM)

0.9431961504529492

In [16]:
len(df_BAM.loc[df_BAM[3] >= 10]) / len(df_BAM)

0.8511950043658303

In [ ]:
Once the current script finishes, delete fastp-trimmed FASTQs for samples that were NOT Nabila's samples (so all those with SAM prefix)


In [3]:
df_Nabila = pd.read_csv("~/Mtb_Megapipe/Nabila_samples.tsv", sep='\t', header=None)
len(df_Nabila)

152

In [4]:
df_Nabila.loc[df_Nabila[0].str.contains('')]

,0,1,2
0,SAMEA104394489,"ERR2199880,ERR2199881",1
1,SAMEA110037905,ERR9807021,1
2,SAMEA11006571,ERR7361908,1
3,SAMEA11006572,ERR7361909,1
4,SAMEA11006573,ERR7361910,1
...,...,...,...
147,TB_TM2152_S7_L001,TB_TM2152_S7_L001,0
148,TB_TM2379_WGS_S54_L001,TB_TM2379_WGS_S54_L001,0
149,TB_TM2857_WGS_S46_L001,TB_TM2857_WGS_S46_L001,0
150,TB_TM2993_WGS_S41_L001,TB_TM2993_WGS_S41_L001,0


In [2]:
df_MIC = pd.read_excel("MIC_03102024_WHO_NI.xlsx", sheet_name=None)
df_pDST = pd.read_excel("phenotypicDST_0310_WHO_NI.xlsx", sheet_name=None)

df_Nabila_FQ = pd.read_csv("samples_with_FQ.csv")
df_Nabila_FQ['FASTQ prefix'] = df_Nabila_FQ['Sample'].str.split('_S').str[0].str.replace('_WGS', '').str.replace('_GCB1', '')

In [3]:
df_MIC.keys(), df_pDST.keys()

(dict_keys(['Instructions of use', 'Accepted drug codes', 'MIC']),
 dict_keys(['Instructions of use', 'Accepted drug codes', 'Accepted row values', 'PDST']))

In [4]:
df_MIC = df_MIC['MIC']
df_pDST = df_pDST['PDST']

In [5]:
col = 'DLM (0.06)'
R_samples = df_pDST.loc[df_pDST[col] == 'R']
len(R_samples)

12

In [6]:
R_samples = R_samples.merge(df_Nabila_FQ, on='FASTQ prefix', how='inner')
print(len(R_samples))

# R_samples.loc[pd.isnull(R_samples['Sample'])]

11


In [8]:
df_Nabila_samples = pd.read_csv("/home/sak0914/Mtb_Megapipe/Nabila_samples.tsv", sep='\t', header=None)
df_Nabila_samples.columns = ['SampleID', 'Run', 'FQ']

In [17]:
df_fastlin_combined = pd.DataFrame(columns=['SampleID', 'mixture', 'lineage', 'lineage_koccur'])

# split lineage from median k-mer occurrence
for i, row in df_Nabila_samples.iterrows():

    sample = row['SampleID']
    run_ids = row['Run'].split(',')

    for run in run_ids:
        
        fName = f"/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/{sample}/{run}/fastlin/output.txt"
        
        if os.path.isfile(fName):
            
            df = pd.read_csv(fName, sep='\t')
    
            if len(df) > 0:
        
                fastlin_lineage = df['lineages'][0]
                mixture = df['mixture'][0]
    
                if mixture == 'yes':
                    print(f"{sample} is mixed")

                if not pd.isnull(fastlin_lineage):
            
                    if ',' not in fastlin_lineage:
                        df_fastlin_combined.loc[i, ['SampleID', 'mixture', 'lineage', 'lineage_koccur']] = [sample, mixture, fastlin_lineage.split(' ')[0], fastlin_lineage.split(' ')[1].replace('(', '').replace(')', '')]
                    else:
                
                        fastlin_lineage_lst = []
                        fastlin_median_occur_lst = []
                        
                        for single_lineage in fastlin_lineage.split(', '):
                            fastlin_lineage_lst.append(single_lineage.split(' ')[0])
                            fastlin_median_occur_lst.append(single_lineage.split(' ')[1].replace('(', '').replace(')', ''))
                
                        # need indices to sort the k-mer occurrences too
                        sorted_idx = np.argsort(fastlin_lineage_lst)
                        sorted_koccur_lst = [fastlin_median_occur_lst[idx] for idx in sorted_idx]
                        
                        df_fastlin_combined.loc[i, ['SampleID', 'mixture', 'lineage', 'lineage_koccur']] = [sample, mixture, ','.join(np.sort(fastlin_lineage_lst)), ','.join(sorted_koccur_lst)]

                else:
                    print(f"NA lineage for {sample}/{run}")
        
        else:
            print(f"No lineage file yet for {sample}/{run}")

df_fastlin_combined['primary_lineage'] = df_fastlin_combined['lineage'].str.split('.').str[0]

SAMEA11006706 is mixed
NA lineage for SAMEA2533665/ERR551055
NA lineage for SAMEA2533665/ERR551056
NA lineage for SAMEA2533665/ERR551057
NA lineage for SAMEA2533757/ERR551179
NA lineage for SAMEA2533757/ERR551180
NA lineage for SAMEA2533757/ERR551181
NA lineage for SAMEA2533757/ERR551182
NA lineage for SAMEA2534556/ERR552304
NA lineage for SAMEA2534556/ERR552305
NA lineage for SAMEA2534556/ERR552306
NA lineage for SAMEA2534556/ERR552307
NA lineage for SAMEA2534891/ERR552768
NA lineage for SAMEA2534891/ERR552769
NA lineage for SAMEA6086630/ERR7451300
NA lineage for SAMEA6086630/ERR7451301
NA lineage for SAMN07344637/SRR5818653
NA lineage for SAMN07344711/SRR5818567
SAMN11179704 is mixed
NA lineage for SAMN12838340/SRR10177263
NA lineage for SAMN12838341/SRR10177262
NA lineage for SAMN12838343/SRR10177260
NA lineage for SAMN12838344/SRR10177259
NA lineage for SAMN12838345/SRR10177258
TB_R47564_WGS_S29_L001 is mixed


In [19]:
df_fastlin_combined.query("SampleID.str.startswith('SAM')")

,SampleID,mixture,lineage,lineage_koccur,primary_lineage
0,SAMEA104394489,no,3,139,3
1,SAMEA110037905,no,1.2.2.1,121,1
2,SAMEA11006571,no,1.2.2.1,56,1
3,SAMEA11006572,no,4.1.2,23,4
4,SAMEA11006573,no,4.3.4.2,107,4
...,...,...,...,...,...
100,SAMN14396398,no,1.1.2,77,1
101,SAMN26553904,no,1.1.1,66,1
102,SAMN41685629,no,1.2.2.2,73,1
103,SAMN29500382,no,2.2.1,276,2


In [22]:
df_WHO_catalog.query("drug_code=='PMD'")

,name,drug_code,range,plate,Medium
5551,SAMEA104394399,PMD,"(0.5,1]",MGIT,MGIT
5552,SAMEA104394489,PMD,"(0.0625,0.125]",MGIT,MGIT
5553,SAMEA104394567,PMD,"(0.0625,0.125]",MGIT,MGIT
5554,SAMEA110037905,PMD,"(1,2]",MGIT,MGIT
5555,SAMEA11006571,PMD,"(0.5,1]",MGIT,MGIT
...,...,...,...,...,...
123391,SAMN13337526,PMD,"(0.25,0.5]",MGIT,MGIT
123392,SAMN13337527,PMD,"(0.5,1]",MGIT,MGIT
123915,SAMN14396398,PMD,"(1,2]",MGIT,MGIT
124530,SAMN26553904,PMD,"(0.5,1]",MGIT,MGIT


In [20]:
df_WHO_catalog = pd.read_csv("~/MtbQuantCNN/MIC_data/2023_WHO_catalog_MIC.csv")

In [15]:
df

,#sample,data_type,k_cov,mixture,lineages,log_barcodes,log_errors
0,ERR551055,paired,21,no,NaN,"4.3.2.1 (14), 4.9 (30)",NaN


In [176]:
for lineage in ['1', '2', '3', '4']:
    num_found = len(df_fastlin_combined.query("lineage.str.startswith(@lineage)"))
    print(f"L{lineage}: {num_found}")

L1: 0
L2: 11
L3: 0
L4: 0


In [100]:
df_Nabila_FQ.query("Sample.str.contains('R39053')")

,Sample,FASTQ prefix


In [58]:
df_Nabila_FQ.query("Sample.str.contains('__')")

,Sample,FASTQ prefix


In [57]:
df_Nabila_FQ.loc[df_Nabila_FQ['Sample'].str.contains('__'), 'Sample'] = df_Nabila_FQ['Sample'].str.replace('__', '_')

In [10]:
df_MIC

,Sample Id,DST Method,FASTQ prefix,DLM,PMD
0,R29153,MGIT,TB_R29153,0.06,<0.25
1,R31038,MGIT,TB_R31038,>0.25,>2
2,R37765GC,MGIT,TB_R37765GC,<0.015,<0.5
3,R36431c,MGIT,TB_R36431c,0.06,0.5
4,R39053,MGIT,TB_R39053,0.06,0.25
5,R37769,MGIT,TB_R37769,<0.015,0.5
6,R37761c,MGIT,TB_R37761c,0.12,0.5
7,R38634,MGIT,TB_R38634,0.03,<0.5
8,R37762,MGIT,TB_R37762,<0.015,0.5
9,R42473,MGIT,TB_R42473,>0.25,>4
